<a href="https://colab.research.google.com/github/elenann-cpu/Privacy-security-trust-ML-systems/blob/main/Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sklearn-pandas

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import LabelEncoder

In [ ]:
import os
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

if not os.path.exists("bank-additional.zip"):
    !wget $url

if not os.path.exists("bank-additional"):
    !unzip -o bank-additional.zip

df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

print(f"Dataset: ({df.shape[0]}, {df.shape[1]})")

df.head(5)

--2026-06-08 13:58:04--  https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank-additional.zip’

bank-additional.zip     [  <=>               ] 434.15K  1.34MB/s    in 0.3s    

2026-06-08 13:58:05 (1.34 MB/s) - ‘bank-additional.zip’ saved [444572]

Archive:  bank-additional.zip
   creating: bank-additional/
  inflating: bank-additional/.DS_Store  
   creating: __MACOSX/
   creating: __MACOSX/bank-additional/
  inflating: __MACOSX/bank-additional/._.DS_Store  
  inflating: bank-additional/.Rhistory  
  inflating: bank-additional/bank-additional-full.csv  
  inflating: bank-additional/bank-additional-names.txt  
  inflating: bank-additional/bank-additional.csv  
  inflating: __MACOSX/._bank-additional  
Dataset: (41188

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


# **Suppression**

In [ ]:
df.drop(columns=["poutcome", "default", "pdays", "duration"], inplace=True)

display(df.head())

print("-" * 30)
print(f"Remaining columns: {len(df.columns)}")
print(f"Dataset: {df.shape}")

,age,job,marital,education,housing,loan,contact,month,day_of_week,campaign,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,telephone,may,mon,1,0,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,no,no,telephone,may,mon,1,0,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,yes,no,telephone,may,mon,1,0,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,telephone,may,mon,1,0,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,yes,telephone,may,mon,1,0,1.1,93.994,-36.4,4.857,5191.0,no


------------------------------
Remaining columns: 17
Dataset: (41188, 17)


# **Generalization**

In [ ]:
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

age_bins = [0, 20, 30, 40, 50, 60, 70, 110]
age_labels = ['<=20', '21-30', '31-40', '41-50', '51-60', '61-70', '>70']
df['age_generalized'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

df['job'] = df['job'].astype(str).str.strip()

job_map = {
    'admin.': 'White-Collar',
    'management': 'White-Collar',
    'technician': 'White-Collar',
    'blue-collar': 'Blue-Collar',
    'services': 'Blue-Collar',
    'housemaid': 'Blue-Collar',
    'entrepreneur': 'Self-Employed',
    'self-employed': 'Self-Employed',
    'retired': 'Not-In-Labor-Force',
    'student': 'Not-In-Labor-Force',
    'unemployed': 'Not-In-Labor-Force',
    'unknown': 'Unknown'
}
df['job_generalized'] = df['job'].map(job_map)

df_generalized = df.drop(columns=['age', 'job'])

df = df_generalized.copy()

display(df[['age_generalized', 'job_generalized']].head())

,age_generalized,job_generalized
0,51-60,Blue-Collar
1,51-60,Blue-Collar
2,31-40,Blue-Collar
3,31-40,White-Collar
4,51-60,Blue-Collar


# **Perturbation**

In [ ]:
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

In [ ]:
# Identify your columns correctly
categorical_cols = ['job']
continuous_cols = ['age']

# Automatically determine 'unchanged' columns
unchanged_cols = [col for col in df.columns if col not in categorical_cols and col not in continuous_cols]

In [ ]:
import scipy.stats as st

def best_fit_distribution(data, bins=200):
    # Only use stable distributions
    dist_names = ['norm', 'gamma', 'lognorm', 'expon']
    best_dist = None
    best_params = None
    min_sse = np.inf

    y, x = np.histogram(data, bins=bins, density=True)
    x = (x + np.roll(x, -1))[:-1] / 2.0

    for name in dist_names:
        dist = getattr(st, name)
        params = dist.fit(data)
        pdf = dist.pdf(x, *params)
        sse = np.sum(np.power(y - pdf, 2.0))
        if sse < min_sse:
            min_sse = sse
            best_dist = name
            best_params = params
    return best_dist, best_params

In [ ]:
def perturb_data(df, unchanged_cols, categorical_cols, continuous_cols, distributions, n):
    """
    Constructs a synthetic/perturbed dataset.
    """
    new_data = {}

    # 1. Copy unchanged columns
    for col in unchanged_cols:
        new_data[col] = np.random.choice(df[col], n)

    # 2. Perturb categorical columns (Sampling with probability)
    for col in categorical_cols:
        counts = df[col].value_counts(normalize=True)
        new_data[col] = np.random.choice(counts.index, p=counts.values, size=n)

    # 3. Perturb continuous columns (Using best-fit distributions)
    for i, col in enumerate(continuous_cols):
        dist_name, params = distributions[i]
        dist = getattr(st, dist_name)
        new_data[col] = dist.rvs(*params, size=n)

    return pd.DataFrame(new_data)

In [ ]:
# Fit distributions for all continuous columns
best_distributions = []
for col in continuous_cols:
    name, params = best_fit_distribution(df[col])
    best_distributions.append((name, params))

gendf = perturb_data(df, unchanged_cols, categorical_cols, continuous_cols, best_distributions, n=len(df))

display(gendf.head())

,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,job,age
0,married,basic.9y,no,yes,no,cellular,aug,thu,252,2,...,0,nonexistent,-1.8,92.963,-46.2,4.960,5195.8,no,retired,45.310009
1,married,basic.6y,unknown,yes,no,cellular,aug,thu,171,1,...,0,nonexistent,-1.8,92.963,-46.2,4.860,5228.1,no,blue-collar,37.566163
2,single,university.degree,no,no,no,cellular,jul,wed,231,4,...,0,nonexistent,-1.8,93.994,-41.8,1.327,5228.1,no,technician,30.147413
3,single,high.school,no,yes,yes,cellular,may,tue,212,11,...,1,failure,-1.8,94.465,-36.4,4.855,5228.1,no,retired,49.091083
4,married,professional.course,unknown,yes,no,telephone,apr,fri,33,4,...,0,nonexistent,-1.8,93.994,-36.4,5.045,5228.1,no,blue-collar,35.433804


# **Anatomization**

In [ ]:
df_anatomization = df.copy()

df_anatomization['group_id'] = df_anatomization.groupby(['education', 'marital']).ngroup()

dataset_1_qi = df_anatomization[['group_id', 'education', 'marital']]

dataset_2_sensitive = df_anatomization[['group_id', 'housing']]

print("DATASET 1: Quasi-Identifiers (Education + Marital)")
display(dataset_1_qi.head())

print("\n" + "-"*50 + "\n")

print("DATASET 2: Sensitive Attributes (Housing Loan Status)")
display(dataset_2_sensitive.head())

DATASET 1: Quasi-Identifiers (Education + Marital)


,group_id,education,marital
0,1,basic.4y,married
1,13,high.school,married
2,13,high.school,married
3,5,basic.6y,married
4,13,high.school,married



--------------------------------------------------

DATASET 2: Sensitive Attributes (Housing Loan Status)


,group_id,housing
0,1,no
1,13,no
2,13,yes
3,5,no
4,13,no


In [ ]:
import matplotlib.patches as patches

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# **k-anonymity**

Listing 8.2 Finding the value span

In [ ]:
categorical = set([
    'job', 'marital', 'education',
    'default', 'housing', 'loan',
    'contact', 'month', 'poutcome'
])

for name in categorical:
    df[name] = df[name].astype('category')

def get_spans(df, partition, scale=None):
    spans = {}

    for column in df.columns:
        col_data = df.loc[partition, column]

        if pd.api.types.is_numeric_dtype(col_data):
            span = col_data.max() - col_data.min()

        else:
            span = col_data.nunique()

        if scale is not None:
            span = span / scale[column]

        spans[column] = span
        print(f"Column: {column}, Span: {span}")

    return spans

full_spans = get_spans(df, df.index)

Column: age, Span: 81
Column: job, Span: 12
Column: marital, Span: 4
Column: education, Span: 8
Column: default, Span: 3
Column: housing, Span: 3
Column: loan, Span: 3
Column: contact, Span: 2
Column: month, Span: 10
Column: day_of_week, Span: 5
Column: duration, Span: 4918
Column: campaign, Span: 55
Column: pdays, Span: 999
Column: previous, Span: 7
Column: poutcome, Span: 3
Column: emp.var.rate, Span: 4.8
Column: cons.price.idx, Span: 2.5660000000000025
Column: cons.conf.idx, Span: 23.9
Column: euribor3m, Span: 4.411
Column: nr.employed, Span: 264.5
Column: y, Span: 2


Listing 8.3 Partitioning the dataset

In [ ]:
feature_columns = ['age', 'education']
sensitive_column = 'y'


def split(df, partition, column):
    dfp = df.loc[partition, column]

    if not pd.api.types.is_numeric_dtype(dfp):
        values = dfp.unique()
        lv = set(values[:len(values)//2])
        rv = set(values[len(values)//2:])
        return dfp.index[dfp.isin(lv)], dfp.index[dfp.isin(rv)]

    else:
        median = dfp.median()
        dfl = dfp.index[dfp < median]
        dfr = dfp.index[dfp >= median]
        return dfl, dfr


def is_k_anonymous(df, partition, sensitive_column, k=3):
    return len(partition) >= k


def partition_dataset(df, feature_columns, sensitive_column, scale, is_valid):

    finished_partitions = []
    partitions = [df.index]

    while partitions:

        partition = partitions.pop(0)

        spans = get_spans(df[feature_columns], partition, scale)

        for column, span in sorted(spans.items(), key=lambda x: -x[1]):

            lp, rp = split(df, partition, column)

            if not is_valid(df, lp, sensitive_column) or not is_valid(df, rp, sensitive_column):
                continue

            partitions.extend((lp, rp))
            break

        else:
            finished_partitions.append(partition)

    return finished_partitions


finished_partitions = partition_dataset(
    df,
    feature_columns,
    sensitive_column,
    full_spans,
    is_k_anonymous
)

print("Number of partitions:", len(finished_partitions))

Column: age, Span: 1.0
Column: education, Span: 1.0
Column: age, Span: 0.24691358024691357
Column: education, Span: 1.0
Column: age, Span: 0.7407407407407407
Column: education, Span: 1.0
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.5
Column: age, Span: 0.24691358024691357
Column: education, Span: 0.5
Column: age, Span: 0.7407407407407407
Column: education, Span: 0.5
Column: age, Span: 0.6666666666666666
Column: education, Span: 0.5
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.24691358024691357
Column: education, Span: 0.25
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.09876543209876543
Column: education, Span: 0.5
Column: age, Span: 0.6296296296296297
Column: education, Span: 0.5
Column: age, Span: 0.09876543209876543
Column: education, Span: 0.5
Column: age, Span: 0.5555555555555556
Column: education, Span: 0.5
C

Listing 8.4 Building the anonymized dataset

In [ ]:
def agg_categorical_column(series):
    return ','.join(set(series))


def agg_numerical_column(series):
    return series.mean()


def build_anonymized_dataset(df, partitions, feature_columns, sensitive_column, max_partitions=None):

    rows = []

    for i, partition in enumerate(partitions):

        if i % 100 == 0:
            print(f"Finished {i} partitions...")

        if max_partitions is not None and i > max_partitions:
            break

        sub_df = df.loc[partition]

        values = {}


        for column in feature_columns:
            if not pd.api.types.is_numeric_dtype(sub_df[column]):
                values[column] = ','.join(set(sub_df[column]))
            else:
                values[column] = sub_df[column].mean()

        sensitive_counts = sub_df[sensitive_column].value_counts()

        for sensitive_value, count in sensitive_counts.items():
            row = values.copy()
            row[sensitive_column] = sensitive_value
            row['count'] = count
            rows.append(row)

    return pd.DataFrame(rows)


dfn = build_anonymized_dataset(
    df,
    finished_partitions,
    feature_columns,
    sensitive_column
)

dfn.head()

Finished 0 partitions...
Finished 100 partitions...
Finished 200 partitions...
Finished 300 partitions...


,age,education,y,count
0,34.000000,illiterate,no,2
1,34.000000,illiterate,yes,1
2,32.495108,professional.course,no,447
3,32.495108,professional.course,yes,64
4,32.465181,university.degree,no,1266


# *l*-diversity

Listing 8.5 Anonymizing the dataset with *l*-diversity

In [ ]:
def agg_categorical_column(series):
    return ','.join(set(series))


def agg_numerical_column(series):
    return series.mean()


def build_anonymized_dataset(df, partitions, feature_columns, sensitive_column, max_partitions=None):

    rows = []

    for i, partition in enumerate(partitions):

        if i % 100 == 0:
            print(f"Finished {i} partitions...")

        if max_partitions is not None and i > max_partitions:
            break

        sub_df = df.loc[partition]

        values = {}


        for column in feature_columns:
            if not pd.api.types.is_numeric_dtype(sub_df[column]):
                values[column] = ','.join(set(sub_df[column]))
            else:
                values[column] = sub_df[column].mean()

        sensitive_counts = sub_df[sensitive_column].value_counts()

        for sensitive_value, count in sensitive_counts.items():
            row = values.copy()
            row[sensitive_column] = sensitive_value
            row['count'] = count
            rows.append(row)

    return pd.DataFrame(rows)


dfn = build_anonymized_dataset(
    df,
    finished_partitions,
    feature_columns,
    sensitive_column
)

dfn.head()

Finished 0 partitions...
Finished 100 partitions...
Finished 200 partitions...
Finished 300 partitions...


,age,education,y,count
0,34.000000,illiterate,no,2
1,34.000000,illiterate,yes,1
2,32.495108,professional.course,no,447
3,32.495108,professional.course,yes,64
4,32.465181,university.degree,no,1266


# t-closeness

Listing 8.6 Checking the frequencies

In [ ]:
global_freqs = {}

total_count = float(len(df))

group_counts = df.groupby('y', observed=False)['y'].count()

for value, count in group_counts.to_dict().items():
    p = count / total_count
    global_freqs[value] = p

print(global_freqs)

{'no': 0.8873458288821987, 'yes': 0.11265417111780131}


Listing 8.7 Anonymizing the dataset with t-closeness

In [ ]:
def t_closeness(df, partition, column, global_freqs):

    total_count = float(len(partition))
    d_max = 0

    group_counts = df.loc[partition][column].value_counts()

    for value, count in group_counts.items():
        p = count / total_count
        d = abs(p - global_freqs.get(value, 0))

        if d > d_max:
            d_max = d

    return d_max


def is_t_close(df, partition, sensitive_column, global_freqs, p=0.2):

    if not isinstance(df[sensitive_column].dtype, pd.CategoricalDtype) and df[sensitive_column].dtype != 'object':
        raise ValueError("t-closeness requires categorical sensitive attribute")

    return t_closeness(df, partition, sensitive_column, global_freqs) <= p


finished_t_close_partitions = partition_dataset(
    df,
    feature_columns,
    sensitive_column,
    full_spans,
    lambda *args: (
        is_k_anonymous(*args) and
        is_t_close(*args, global_freqs)
    )
)


dft = build_anonymized_dataset(
    df,
    finished_t_close_partitions,
    feature_columns,
    sensitive_column
)


column_x, column_y = feature_columns[:2]

print(dft.sort_values([column_x, column_y, sensitive_column]))
dft.head()

Column: age, Span: 1.0
Column: education, Span: 1.0
Column: age, Span: 0.24691358024691357
Column: education, Span: 1.0
Column: age, Span: 0.7407407407407407
Column: education, Span: 1.0
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.5
Column: age, Span: 0.24691358024691357
Column: education, Span: 0.5
Column: age, Span: 0.7407407407407407
Column: education, Span: 0.5
Column: age, Span: 0.6666666666666666
Column: education, Span: 0.5
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.24691358024691357
Column: education, Span: 0.25
Column: age, Span: 0.2345679012345679
Column: education, Span: 0.25
Column: age, Span: 0.09876543209876543
Column: education, Span: 0.5
Column: age, Span: 0.6296296296296297
Column: education, Span: 0.5
Column: age, Span: 0.09876543209876543
Column: education, Span: 0.5
Column: age, Span: 0.5555555555555556
Column: education, Span: 0.5
C

,age,education,y,count
0,24.563559,unknown,no,169
1,24.563559,unknown,yes,67
2,43.833333,illiterate,no,5
3,43.833333,illiterate,yes,1
4,32.495108,professional.course,no,447
